In [1]:
# %load_ext autoreload
# %autoreload 2

# # autoreload re-imports external.raster_to_graph modules from disk before every cell run,
# # so edits to run_inference_generous_phase4.py / filters / scoring take effect without
# # restarting the kernel. The model checkpoint itself is cached in _MODEL_CACHE —
# # that's intentional; only the inference code should reload, not the weights.

# Phase 4 — Raster-to-Graph single-image inference

Set `INPUT_IMAGE_PATH` in the last cell to any floor plan image, then **Run All Cells**.

This runs: true-20%-pad preprocessing → SizheHu pretrained Deformable-DETR autoregressive
inference (4 Monte Carlo attempts + mask-and-rerun multi-start) → component merge
(node snap + intersection split + collinear merge) → light post-merge filter, and writes
the predicted wall graph alongside the input image:

- `input.png` — preprocessed 512×512 canvas
- `graph_pred.json` — `{nodes: [[x,y], ...], edges: [[x1,y1,x2,y2], ...]}`
- `graph_pred.svg` — SVG vector overlay
- `graph_overlay.png` — final graph on preprocessed image (red nodes, blue edges)
- `graph_overlay_components.png` — per-component coloured, before merge
- `graph_overlay_merged.png` — after merge, before light filter
- `components.json` — per-component detail
- `metrics.json` — stage counts, soft scores, preprocessing metrics

In [2]:
from __future__ import annotations
from PIL import Image  # must come before any torchvision import (PIL DLL ordering quirk on Windows)

import gc
import json
import sys
import time
from pathlib import Path

import torch


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "external" / "raster_to_graph").exists() and (candidate / "configs").exists():
            return candidate
    return here


PROJECT_ROOT = _find_project_root()
EXTERNAL_DIR = PROJECT_ROOT / "external" / "raster_to_graph"
CHECKPOINT   = PROJECT_ROOT / "checkpoints_Raster2Graph" / "checkpoint0299.pth"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(EXTERNAL_DIR) not in sys.path:
    sys.path.insert(0, str(EXTERNAL_DIR))

from args import get_args_parser
from models.build import build_model, build_postprocessor
from util.random_utils import set_random_seed

from run_inference_generous_phase4 import (
    GENEROUS, MASK_RERUN, MERGE, FILTERS, SCORING,
    preprocess_crop512_margin20_truepad,
    run_generous_multistart,
    merge_components,
    apply_light_post_merge_filter,
    compute_soft_scores,
    compute_candidate_score,
    make_svg_merged,
    make_overlay_normal,
    make_overlay_components,
    _find_components,
    _agg_filter_stats,
)

import numpy as np

_MODEL_CACHE = None


def _load_model():
    global _MODEL_CACHE
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE
    if not CHECKPOINT.exists():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available — this pipeline requires a GPU.")

    gc.collect()
    torch.cuda.empty_cache()

    device = torch.device("cuda")
    args_p = get_args_parser()
    args   = args_p.parse_args([])
    args.device = "cuda"
    set_random_seed(args)

    model = build_model(args).to(device)
    postprocessor = build_postprocessor()
    ckpt = torch.load(str(CHECKPOINT), map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.eval()

    _MODEL_CACHE = (model, postprocessor, device, ckpt["epoch"])
    print(f"  Loaded checkpoint epoch {ckpt['epoch']} onto {device}")
    return _MODEL_CACHE


def raster2graph_image(image_path):
    """Run phase4 raster-to-graph inference on a single floor plan image.

    Args:
        image_path: Path to the input raster image (any size/format).

    Saves beside the input image:
        input.png                     preprocessed 512x512 canvas
        graph_pred.json               predicted wall graph {nodes, edges}
        graph_pred.svg                SVG overlay
        graph_overlay.png             final graph on input (red nodes, blue edges)
        graph_overlay_components.png  per-component coloured, before merge
        graph_overlay_merged.png      after merge, before light filter
        components.json               per-component detail
        metrics.json                  stage counts, soft scores, preprocessing info

    Returns:
        Path to graph_pred.json.
    """
    image_path = Path(image_path).expanduser().resolve()
    if not image_path.exists():
        raise FileNotFoundError(f"Input image not found: {image_path}")

    out_dir = image_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    sid = image_path.stem

    model, postprocessor, device, ckpt_epoch = _load_model()

    t0 = time.perf_counter()

    # Preprocessing: true 20% white-canvas padding -> 512x512
    base_pil, preproc_metrics = preprocess_crop512_margin20_truepad(image_path)
    touches = preproc_metrics["content_touches_edge"]
    margins = preproc_metrics["final_canvas_margins_px"]
    if touches:
        print(f"  [WARN] content touches canvas edge after true-pad preprocessing")
    else:
        print(f"  margins: L={margins['left']} T={margins['top']} "
              f"R={margins['right']} B={margins['bottom']} px")

    gray_arr = np.asarray(base_pil.convert("L"))

    # Autoregressive inference: MC x4 + mask-and-rerun multi-start
    accepted, discarded = run_generous_multistart(
        model, postprocessor, base_pil, device, GENEROUS, MASK_RERUN
    )

    comp_node_count = sum(c["num_points"] for c in accepted)
    comp_edge_count = sum(c["num_edges"]  for c in accepted)

    # Merge: node snap + intersection split + collinear merge
    merged_pts_raw, merged_edges_raw = merge_components(
        accepted,
        snap_tol  = MERGE["node_snap_tolerance_px"],
        inter_tol = MERGE["edge_intersection_tolerance_px"],
        col_tol   = MERGE["collinear_overlap_tolerance_px"],
    )
    merged_raw_node_count = len(merged_pts_raw)
    merged_raw_edge_count = len(merged_edges_raw)

    # Light post-merge filter: angle violations + dedup only
    merged_pts, merged_edges, light_fstats = apply_light_post_merge_filter(
        merged_pts_raw, merged_edges_raw, gray_arr, FILTERS
    )

    merged_scores = compute_soft_scores(merged_pts, merged_edges, gray_arr, SCORING)
    elapsed = time.perf_counter() - t0
    empty   = len(merged_pts) == 0
    agg_fstats = _agg_filter_stats(accepted)

    # Write all outputs
    base_pil.save(str(out_dir / "input.png"))

    graph_json = {
        "nodes": [[int(x), int(y)] for x, y in merged_pts],
        "edges": [[int(p1[0]), int(p1[1]), int(p2[0]), int(p2[1])] for p1, p2 in merged_edges],
    }
    json_path = out_dir / "graph_pred.json"
    json_path.write_text(json.dumps(graph_json, indent=2), encoding="utf-8")
    (out_dir / "graph_pred.svg").write_text(make_svg_merged(merged_pts, merged_edges), encoding="utf-8")

    make_overlay_components(base_pil, accepted).save(str(out_dir / "graph_overlay_components.png"))
    make_overlay_normal(base_pil, merged_pts_raw, merged_edges_raw).save(str(out_dir / "graph_overlay_merged.png"))
    make_overlay_normal(base_pil, merged_pts, merged_edges).save(str(out_dir / "graph_overlay.png"))

    comp_out = {
        "sample_id": sid,
        "components_before_merge": [
            {
                "component_id":     c["component_id"],
                "source":           c["source"],
                "num_points":       c["num_points"],
                "num_edges":        c["num_edges"],
                "stop_code":        c["stop_code"],
                "candidate_score":  c["candidate_score"],
                "selected_attempt": c["selected_attempt"],
                "filter_stats":     c["filter_stats"],
                "scores": {k: v for k, v in c["scores"].items() if k != "per_edge_evidence"},
                "accepted": True,
            }
            for c in accepted
        ],
        "discarded_components": [
            {
                "component_id": c["component_id"],
                "source":       c["source"],
                "num_points":   c["num_points"],
                "num_edges":    c["num_edges"],
                "stop_code":    c["stop_code"],
                "accepted":     False,
            }
            for c in discarded
        ],
        "merged_graph": {
            "num_nodes_after_merge":        merged_raw_node_count,
            "num_edges_after_merge":        merged_raw_edge_count,
            "num_nodes_after_light_filter": len(merged_pts),
            "num_edges_after_light_filter": len(merged_edges),
            "light_post_merge_filter_stats": light_fstats,
        },
    }
    (out_dir / "components.json").write_text(json.dumps(comp_out, indent=2), encoding="utf-8")

    metrics = {
        "sample_id":              sid,
        "source_image":           str(image_path),
        "source_variant":         preproc_metrics["source_variant"],
        "standardized_margin":    preproc_metrics["standardized_margin"],
        "content_bbox_original":  preproc_metrics["content_bbox_original"],
        "content_bbox_after_preprocess": preproc_metrics["content_bbox_after_preprocess"],
        "final_canvas_margins_px": preproc_metrics["final_canvas_margins_px"],
        "content_touches_edge":   touches,
        "checkpoint_epoch":       ckpt_epoch,
        "stage_counts": {
            "components_nodes": comp_node_count,
            "components_edges": comp_edge_count,
            "merged_nodes":     merged_raw_node_count,
            "merged_edges":     merged_raw_edge_count,
            "final_nodes":      len(merged_pts),
            "final_edges":      len(merged_edges),
        },
        "soft_scores": {
            "wall_evidence_alignment_score": merged_scores["wall_evidence_alignment_score"],
            "rectangle_cycle_count":         merged_scores["rectangle_cycle_count"],
            "rectangle_cycle_score":         merged_scores["rectangle_cycle_score"],
            "dangling_node_count":           merged_scores["dangling_node_count"],
            "dangling_penalty":              merged_scores["dangling_penalty"],
            "unsupported_edge_ratio":        merged_scores["unsupported_edge_ratio"],
            "small_component_count":         merged_scores["small_component_count"],
            "small_component_penalty":       merged_scores["small_component_penalty"],
            "candidate_score": compute_candidate_score(
                merged_scores, len(merged_edges), max(len(merged_edges), 1), SCORING
            ),
        },
        "components_before_merge": len(accepted),
        "components_after_merge":  len(_find_components(merged_pts, merged_edges)),
        "final_num_points":        len(merged_pts),
        "final_num_edges":         len(merged_edges),
        "empty":                   empty,
        "elapsed_s":               round(elapsed, 2),
    }
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    if empty:
        note = "Empty — no significant component survived filters."
    elif len(accepted) > 1:
        note = (f"{len(accepted)} components. "
                f"Components: {comp_node_count}pts/{comp_edge_count}edges -> "
                f"Merged: {merged_raw_node_count}pts/{merged_raw_edge_count}edges -> "
                f"Final: {len(merged_pts)}pts/{len(merged_edges)}edges. "
                f"wall_ev={merged_scores['wall_evidence_alignment_score']:.2f}.")
    else:
        note = (f"1 component. "
                f"Components: {comp_node_count}pts/{comp_edge_count}edges -> "
                f"Merged: {merged_raw_node_count}pts/{merged_raw_edge_count}edges -> "
                f"Final: {len(merged_pts)}pts/{len(merged_edges)}edges. "
                f"wall_ev={merged_scores['wall_evidence_alignment_score']:.2f}.")
    (out_dir / "notes.txt").write_text(note + "\n", encoding="utf-8")

    summary = {
        "input":               str(image_path),
        "output_dir":          str(out_dir),
        "graph_json":          str(json_path),
        "graph_svg":           str(out_dir / "graph_pred.svg"),
        "graph_overlay":       str(out_dir / "graph_overlay.png"),
        "final_nodes":         len(merged_pts),
        "final_edges":         len(merged_edges),
        "wall_evidence":       merged_scores["wall_evidence_alignment_score"],
        "rectangle_cycles":    merged_scores["rectangle_cycle_count"],
        "components_accepted": len(accepted),
        "empty":               empty,
        "elapsed_s":           round(elapsed, 2),
        "checkpoint_epoch":    ckpt_epoch,
        "device":              str(device),
    }
    print(json.dumps(summary, indent=2))
    return json_path

In [3]:
# <<< EDIT THIS, then Run All Cells >>>
INPUT_IMAGE_PATH = r"C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\outputs\vectorization\phase4_raster2graph_generous_inference\sample_003\image.png"

raster2graph_image(INPUT_IMAGE_PATH)

c:\Users\kdgki\anaconda3\envs\floorplan-cad\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\kdgki\anaconda3\envs\floorplan-cad\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  Loaded checkpoint epoch 299 onto cuda
  margins: L=73 T=129 R=73 B=129 px


c:\Users\kdgki\anaconda3\envs\floorplan-cad\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


{
  "input": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_raster2graph_generous_inference\\sample_003\\image.png",
  "output_dir": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_raster2graph_generous_inference\\sample_003",
  "graph_json": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_raster2graph_generous_inference\\sample_003\\graph_pred.json",
  "graph_svg": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_raster2graph_generous_inference\\sample_003\\graph_pred.svg",
  "graph_overlay": "C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_raster2graph_generous_inference\\sample_003\\graph_overlay.png",
  "final_nodes": 24,
  "final_edges": 15,
  "wall_evidence": 0.362,
  "rectangle_cycles": 0,
  "components_accepted": 1,
  "empty": false,
  "elapsed_s": 5.08

WindowsPath('C:/Users/kdgki/Desktop/MSCDP/Projects/neural_floorplan/outputs/vectorization/phase4_raster2graph_generous_inference/sample_003/graph_pred.json')